# Phase 0 — ComfyUI Environment Setup (Colab)

**Purpose:** Install ComfyUI + all required custom nodes + download base models to Google Drive.
Run this once per Colab session (or skip model downloads if Drive already has them).

**Runtime:** A100 (40 GB) recommended. T4 (16 GB) works for LTX-Video 2B + SDXL, not for Wan 14B models.

**Steps:**
1. Mount Drive
2. Check GPU
3. Install ComfyUI
4. Install custom nodes
5. Download models (first time only)
6. Expose ComfyUI via cloudflared tunnel
7. Health check

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# All persistent data lives here — models, LoRAs, outputs, character library
DRIVE_BASE = '/content/drive/MyDrive/ai_character_studio'
MODELS_DIR = f'{DRIVE_BASE}/models'
LORAS_DIR  = f'{DRIVE_BASE}/loras'
CHARS_DIR  = f'{DRIVE_BASE}/characters'
COMFY_DIR  = '/content/ComfyUI'  # ephemeral (re-cloned each session)

import os
for d in [DRIVE_BASE, MODELS_DIR, LORAS_DIR, CHARS_DIR,
          f'{MODELS_DIR}/checkpoints', f'{MODELS_DIR}/vae',
          f'{MODELS_DIR}/controlnet', f'{MODELS_DIR}/ipadapter',
          f'{MODELS_DIR}/clip_vision',    # CLIP Vision for IPAdapter FaceID
          f'{MODELS_DIR}/clip',           # T5 text encoder for LTX-Video
          f'{MODELS_DIR}/video', f'{MODELS_DIR}/upscale']:
    os.makedirs(d, exist_ok=True)

print('Drive mounted. Studio root:', DRIVE_BASE)

## 2. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 3. Install ComfyUI

In [ ]:
import subprocess, sys

if not os.path.exists(COMFY_DIR):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}
else:
    !git -C {COMFY_DIR} pull --ff-only

!pip install -q -r {COMFY_DIR}/requirements.txt
!pip install -q huggingface_hub diffusers transformers accelerate safetensors

# Fix: kornia version shipped with Colab breaks ComfyUI-LTXVideo pyramid blending
!pip install -q --upgrade kornia

# Fix: InsightFace required by IPAdapterInsightFaceLoader (IPAdapter FaceID)
!pip install -q insightface onnxruntime

# Note: comfy-aimdo is a hard dependency of ComfyUI — cannot be removed.
# It auto-downloads Qwen 3 4B on first run. We redirect its cache to Drive
# so the model persists and never re-downloads after the first session.
import os
aimdo_cache = f'{DRIVE_BASE}/models/comfy_aimdo_cache'
os.makedirs(aimdo_cache, exist_ok=True)
os.environ['COMFY_AIMDO_CACHE'] = aimdo_cache
os.environ['HF_HOME'] = aimdo_cache  # HuggingFace cache → Drive
print(f'comfy-aimdo cache → Drive: {aimdo_cache}')

print('ComfyUI installed with all dependency fixes applied.')

## 4. Install Custom Nodes

In [ ]:
NODES_DIR = f'{COMFY_DIR}/custom_nodes'
os.makedirs(NODES_DIR, exist_ok=True)

def install_node(repo, name=None):
    name = name or repo.rstrip('/').split('/')[-1]
    dest = f'{NODES_DIR}/{name}'
    if not os.path.exists(dest):
        print(f'  Installing {name}...')
        result = subprocess.run(['git', 'clone', '--depth', '1', repo, dest],
                                capture_output=True, text=True)
        if result.returncode != 0:
            print(f'  WARN: {result.stderr[:200]}')
            return
        req = f'{dest}/requirements.txt'
        if os.path.exists(req):
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', req])
    else:
        print(f'  ✓ {name}')

nodes = [
    # Video helper (VHS) — loads/saves video in ComfyUI
    'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite',
    # ControlNet preprocessors (DWPose, Depth Anything, etc.)
    'https://github.com/Fannovel16/comfyui_controlnet_aux',
    # Advanced ControlNet (multi-stack)
    'https://github.com/Kosinkadink/ComfyUI-Advanced-ControlNet',
    # IP-Adapter+
    'https://github.com/cubiq/ComfyUI_IPAdapter_plus',
    # LTX-Video (official Lightricks)
    'https://github.com/Lightricks/ComfyUI-LTXVideo',
    # Wan Video Wrapper (VACE, Animate, FLF2V)
    'https://github.com/kijai/ComfyUI-WanVideoWrapper',
    # SAM 2 (segmentation for Mode 2 VACE path)
    'https://github.com/kijai/ComfyUI-segment-anything-2',
    # Impact Pack (face detection)
    'https://github.com/ltAstrid/ComfyUI-Impact-Pack',
    # Tooling / utility nodes
    'https://github.com/Acly/comfyui-tooling-nodes',
    'https://github.com/pythongosssss/ComfyUI-Custom-Scripts',
]

print('Installing custom nodes...')
for repo in nodes:
    install_node(repo)
print('Done.')

## 5. Point ComfyUI at Drive models

In [ ]:
extra_paths = f"""
character_studio:
  base_path: {MODELS_DIR}
  checkpoints: checkpoints/
  loras: loras/
  vae: vae/
  controlnet: controlnet/
  ipadapter: ipadapter/
  clip_vision: clip_vision/
  clip: clip/
  video_models: video/
  diffusion_models: video/
  upscale_models: upscale/

wan_vaes:
  base_path: {MODELS_DIR}/video
  vae: Wan2.1-VACE-1.3B/

trained_loras:
  base_path: {DRIVE_BASE}
  loras: loras/
"""
# Notes:
# - diffusion_models mirrors video/ so WanVideoModelLoader + LTX UNETLoader both find models
# - wan_vaes secondary section makes WanVideoVAELoader find Wan2.1_VAE.pth
# - trained_loras section maps the top-level loras/ dir where 01b saves character LoRAs
#   (01b writes to DRIVE_BASE/loras, ComfyUI's primary loras path is DRIVE_BASE/models/loras)
# - Google Drive FUSE does not support symlinks, so secondary yaml sections are used instead
# - clip/ holds the LTX-Video T5-XXL text encoder (ltxv_t5xxl.safetensors)
with open(f'{COMFY_DIR}/extra_model_paths.yaml', 'w') as f:
    f.write(extra_paths)
print('extra_model_paths.yaml written (incl. trained_loras path for 01b outputs).')

## 6. Download Base Models (first time only)
Skip individual cells if the file already exists on Drive.

In [ ]:
from huggingface_hub import hf_hub_download
import shutil

def dl(repo_id, filename, dest_dir, dest_name=None):
    dest_name = dest_name or filename.split('/')[-1]
    dest_path = f'{dest_dir}/{dest_name}'
    if os.path.exists(dest_path):
        print(f'  ✓ {dest_name} (already on Drive)')
        return dest_path
    print(f'  ↓ {dest_name} ...')
    tmp = hf_hub_download(repo_id=repo_id, filename=filename)
    shutil.copy2(tmp, dest_path)
    print(f'  ✓ {dest_name} saved to Drive')
    return dest_path

# SDXL base + VAE
dl('stabilityai/stable-diffusion-xl-base-1.0',
   'sd_xl_base_1.0.safetensors',
   f'{MODELS_DIR}/checkpoints')

dl('madebyollin/sdxl-vae-fp16-fix',
   'sdxl_vae.safetensors',
   f'{MODELS_DIR}/vae')

print('SDXL base models ready.')

In [ ]:
# ControlNet — SDXL OpenPose + Depth
dl('thibaud/controlnet-openpose-sdxl-1.0',
   'OpenPoseXL2.safetensors',
   f'{MODELS_DIR}/controlnet', 'sdxl_openpose.safetensors')

dl('diffusers/controlnet-depth-sdxl-1.0',
   'diffusion_pytorch_model.fp16.safetensors',
   f'{MODELS_DIR}/controlnet', 'sdxl_depth_fp16.safetensors')

# IP-Adapter FaceID Plus v2 SDXL
dl('h94/IP-Adapter-FaceID',
   'ip-adapter-faceid-plusv2_sdxl.bin',
   f'{MODELS_DIR}/ipadapter')
dl('h94/IP-Adapter-FaceID',
   'ip-adapter-faceid-plusv2_sdxl_lora.safetensors',
   f'{MODELS_DIR}/ipadapter')

# CLIP Vision models for IPAdapter FaceID
# IMPORTANT: ip-adapter-faceid-plusv2_sdxl.bin needs ViT-H (1280 dims), NOT ViT-bigG (1664 dims)
# ViT-H — required by FaceID Plus V2 SDXL (1280 feature dims)
dl('h94/IP-Adapter',
   'models/image_encoder/model.safetensors',
   f'{MODELS_DIR}/clip_vision',
   'CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors')

# ViT-bigG — for standard IPAdapter SDXL (not FaceID); download for completeness
dl('h94/IP-Adapter',
   'sdxl_models/image_encoder/model.safetensors',
   f'{MODELS_DIR}/clip_vision',
   'CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors')

# InsightFace Python package — required by IPAdapterInsightFaceLoader
!pip install -q insightface onnxruntime

# LTX-Video VAE — separate from the main model safetensors file
dl('Lightricks/LTX-Video',
   'vae/diffusion_pytorch_model.safetensors',
   f'{MODELS_DIR}/vae', 'ltxv_vae.safetensors')

# Symlink CLIP Vision models into ComfyUI's LOCAL clip_vision folder.
# ComfyUI's extra_model_paths maps clip_vision to Drive, but local symlinks
# ensure models appear in the dropdown immediately without needing a restart.
# (Drive files can appear slow or not refresh mid-session)
import subprocess
local_cv = f'{COMFY_DIR}/models/clip_vision'
os.makedirs(local_cv, exist_ok=True)
for fn in ['CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors',
           'CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors']:
    src  = f'{MODELS_DIR}/clip_vision/{fn}'
    dest = f'{local_cv}/{fn}'
    if os.path.exists(src) and not os.path.exists(dest):
        subprocess.run(['ln', '-sf', src, dest])
        print(f'Symlinked CLIP Vision: {fn}')

print('ControlNet + IPAdapter + CLIP Vision + LTX VAE ready.')

In [ ]:
# LTX-Video models (verified filenames Sept 2026)
# Naming changed in v0.9.6+: old "ltx-video-2b-v0.9.safetensors" is now "ltxv-*"

# 2B distilled FP8 — 4.5 GB, fits T4, fastest inference, multi-keyframe (recommended default)
dl('Lightricks/LTX-Video',
   'ltxv-2b-0.9.8-distilled-fp8.safetensors',
   f'{MODELS_DIR}/video')

# Spatial upscaler — 505 MB, worth having for quality
dl('Lightricks/LTX-Video',
   'ltxv-spatial-upscaler-0.9.8.safetensors',
   f'{MODELS_DIR}/video')

# LTX-Video T5 text encoder.
# The official repo shards it into 4 × ~5 GB files (~19 GB total).
# Instead use the community FP8-compressed single file (~5 GB) from comfyanonymous/flux_text_encoders.
# This file also works with FLUX and other T5-XXL models — one download covers all.
dl('comfyanonymous/flux_text_encoders',
   't5xxl_fp8_e4m3fn.safetensors',
   f'{MODELS_DIR}/clip', 'ltxv_t5xxl_fp8.safetensors')

# 13B dev FP8 — 15.7 GB, A100 needed, best quality (uncomment on A100)
# dl('Lightricks/LTX-Video', 'ltxv-13b-0.9.8-dev-fp8.safetensors', f'{MODELS_DIR}/video')

# Symlink T5 to local text_encoders so CLIPLoader picks it up without restart
import subprocess, os
t5_src  = f'{MODELS_DIR}/clip/ltxv_t5xxl_fp8.safetensors'
t5_dest = f'{COMFY_DIR}/models/text_encoders/ltxv_t5xxl_fp8.safetensors'
if os.path.exists(t5_src) and not os.path.exists(t5_dest):
    subprocess.run(['ln', '-sf', t5_src, t5_dest])
    print('T5 symlinked to local text_encoders/')

print('LTX-Video models ready.')

In [ ]:
# Wan models (verified Sept 2026)
# Both repos use sharded weights stored as diffusers-format folders
# (not a single .safetensors) — must use snapshot_download, not hf_hub_download

from huggingface_hub import snapshot_download

def dl_snapshot(repo_id, dest_dir):
    dest = f'{dest_dir}/{repo_id.split("/")[-1]}'
    if os.path.isdir(dest) and os.listdir(dest):
        print(f'  ✓ {repo_id} (already on Drive)')
        return dest
    print(f'  ↓ {repo_id} (snapshot download, may take a while)...')
    snapshot_download(repo_id=repo_id, local_dir=dest,
                      ignore_patterns=['*.msgpack', '*.h5', 'flax_model*'])
    print(f'  ✓ {repo_id} saved to {dest}')
    return dest

# Wan2.1 VACE 1.3B — 480p editing/swap, ~7 GB, works on T4
# Stores as: diffusion_pytorch_model.safetensors (+ config files)
dl_snapshot('Wan-AI/Wan2.1-VACE-1.3B', f'{MODELS_DIR}/video')

# Wan2.1 FLF2V 14B — 720p first+last frame, ~65 GB total (7 shards × ~10 GB each)
# Needs A100 40 GB minimum with offloading. Only download if you have the space.
# dl_snapshot('Wan-AI/Wan2.1-FLF2V-14B-720P', f'{MODELS_DIR}/video')

# Wan2.2 Animate 14B — character replacement / pose-driven animation
# Needs A100 80 GB or A100 40 GB with heavy offloading.
# dl_snapshot('Wan-AI/Wan2.2-Animate-14B', f'{MODELS_DIR}/video')

print('Wan models ready (heavy 14B variants commented — uncomment on A100 40/80 GB).')

In [ ]:
# SeedVR2 3B upscaler (optional final pass)
# dl('ByteDance-Seed/SeedVR', '...', f'{MODELS_DIR}/upscale')
print('Upscale model: uncomment above when ready for final-pass quality.')

## 7. Start ComfyUI + expose via tunnel

In [ ]:
# Install cloudflared for public tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import os, subprocess, urllib.request

WORKFLOWS_DIR = f'{COMFY_DIR}/user/default/workflows'
REPO_RAW = 'https://raw.githubusercontent.com/marcdemory-8451/ai-character-studio/main'

# Create ComfyUI directory structure
for d in [
    f'{COMFY_DIR}/user/default/workflows',
    f'{COMFY_DIR}/user/default/user_settings',
    f'{COMFY_DIR}/models/text_encoders',
    f'{COMFY_DIR}/models/clip_vision',
    f'{COMFY_DIR}/web/extensions',
]:
    os.makedirs(d, exist_ok=True)

# Download workflow JSONs from GitHub into ComfyUI's workflow browser
# These appear under Workflows → Browse in the ComfyUI UI
workflows = [
    'engine/comfyui/workflows/02_stills_ipadapter_faceid.json',
    'engine/comfyui/workflows/03_video_mode1_ltx_flf2v.json',
    'engine/comfyui/workflows/04_video_wan_vace_animate.json',
]
for wf_path in workflows:
    dest = f'{WORKFLOWS_DIR}/{os.path.basename(wf_path)}'
    if not os.path.exists(dest):
        try:
            urllib.request.urlretrieve(f'{REPO_RAW}/{wf_path}', dest)
            print(f'  ↓ {os.path.basename(wf_path)}')
        except Exception as e:
            print(f'  WARN: could not download {wf_path}: {e}')
    else:
        print(f'  ✓ {os.path.basename(wf_path)}')

# Install splash_fix custom node (auto-dismisses stuck splash after 15s)
splash_node_dir = f'{COMFY_DIR}/custom_nodes/splash_fix_node'
os.makedirs(f'{splash_node_dir}/web/js', exist_ok=True)
with open(f'{splash_node_dir}/__init__.py', 'w') as f:
    f.write('NODE_CLASS_MAPPINGS = {}\nNODE_DISPLAY_NAME_MAPPINGS = {}\nWEB_DIRECTORY = "./web/js"\n')
with open(f'{splash_node_dir}/web/js/splash_fix.js', 'w') as f:
    f.write("""import { app } from "../../scripts/app.js";
app.registerExtension({
  name: "SplashFix",
  async setup() {
    setTimeout(() => {
      const s = document.getElementById("splash-loader");
      if (s) {
        console.warn("[SplashFix] Force-dismissing stuck splash");
        s.style.transition = "opacity 0.6s"; s.style.opacity = "0";
        setTimeout(() => s.remove(), 650);
      }
    }, 15000);
  },
});
""")

# Symlink Wan T5 encoder into local text_encoders/
wan_t5_src  = f'{MODELS_DIR}/video/Wan2.1-VACE-1.3B/models_t5_umt5-xxl-enc-bf16.pth'
wan_t5_dest = f'{COMFY_DIR}/models/text_encoders/models_t5_umt5-xxl-enc-bf16.pth'
if os.path.exists(wan_t5_src) and not os.path.exists(wan_t5_dest):
    subprocess.run(['ln', '-sf', wan_t5_src, wan_t5_dest])

# Symlink LTX T5 if downloaded
ltx_t5_src  = f'{MODELS_DIR}/clip/ltxv_t5xxl_fp8.safetensors'
ltx_t5_dest = f'{COMFY_DIR}/models/text_encoders/ltxv_t5xxl_fp8.safetensors'
if os.path.exists(ltx_t5_src) and not os.path.exists(ltx_t5_dest):
    subprocess.run(['ln', '-sf', ltx_t5_src, ltx_t5_dest])

# Symlink CLIP Vision models to local folder
local_cv = f'{COMFY_DIR}/models/clip_vision'
for fn in ['CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors',
           'CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors',
           'CLIP-ViT-bigG-14-laion2B-39B-b160k.bin']:
    src  = f'{MODELS_DIR}/clip_vision/{fn}'
    dest = f'{local_cv}/{fn}'
    if os.path.exists(src) and not os.path.exists(dest):
        subprocess.run(['ln', '-sf', src, dest])

print('Setup complete: user dirs, workflows imported, splash fix, T5 & CLIP Vision symlinks, cloudflared.')

In [ ]:
import threading, subprocess, time, os

# Redirect comfy-aimdo / HuggingFace cache to Drive so Qwen 3 4B persists between sessions
AIMDO_CACHE = f'{DRIVE_BASE}/models/comfy_aimdo_cache'
os.makedirs(AIMDO_CACHE, exist_ok=True)
os.environ['HF_HOME'] = AIMDO_CACHE
os.environ['COMFY_AIMDO_CACHE'] = AIMDO_CACHE

# Start ComfyUI in background
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188',
     '--preview-method', 'auto', '--disable-xformers'],
    cwd=COMFY_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    env={**os.environ}
)
time.sleep(8)

# Start cloudflared with --protocol http2 — critical for WebSocket support
# QUIC (the default) breaks WebSocket upgrade causing the splash to hang forever
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8188', '--protocol', 'http2'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

# Extract tunnel URL
tunnel_url = None
for _ in range(30):
    line = tunnel_proc.stdout.readline().decode('utf-8', errors='ignore')
    if 'trycloudflare.com' in line:
        import re
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if m:
            tunnel_url = m.group(0)
            break

if tunnel_url:
    print(f'\n✅ ComfyUI is live at: {tunnel_url}')
    print(f'   comfy-aimdo cache → {AIMDO_CACHE}')
    print(f'   WebSocket: cloudflared --protocol http2 (fixes splash hang)')
else:
    print('ComfyUI started but tunnel URL not found. Check Colab output.')

## 8. Health check

In [ ]:
import urllib.request, json
try:
    with urllib.request.urlopen('http://localhost:8188/system_stats') as r:
        stats = json.loads(r.read())
    print('ComfyUI is responding.')
    print('System stats:', json.dumps(stats, indent=2))
except Exception as e:
    print(f'ComfyUI not responding: {e}. Check the startup output above.')

---
## ✅ Setup complete

**Next notebooks:**
- `01a_caption_refs.ipynb` — auto-caption reference images
- `01b_train_sdxl_lora.ipynb` — train SDXL character LoRA
- `02_test_stills.ipynb` — test stills generation via ComfyUI API
- `03_test_video_mode1.ipynb` — test LTX-Video keyframe interpolation
- `04_test_video_mode2.ipynb` — test Wan VACE / Animate character replacement
